In this file we will be using mixed precision training 

In [1]:
import pandas as pd
import regex as re
import torch

In [2]:
df = pd.read_csv(r'D:\Traffic\labels_processed.csv')

In [3]:
def label_function(dpath):
    class_name = re.findall(r'(\d+)_.*\.png$', dpath.name)
    class_id = int(class_name[0])
    return class_id

In [4]:
from pathlib import Path

In [5]:
path = Path(r'D:\Traffic\traffic_Data_processed\DATA')

In [6]:
lr_head = 0.0017378008365631102
lr_whole_model = 1.9054607491852948e-06

In [7]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [8]:
from torch.utils.data import Dataset
from PIL import Image

In [9]:
class dset(Dataset):
    def __init__(self, image_paths, transform = None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        label = label_function(Path(image_path))
        if self.transform:
            image = self.transform(image)
        return image, label

In [10]:
image_paths = list(path.rglob("*.png"))

In [11]:
from torchvision import transforms
from torch.utils.data import random_split
from torch.utils.data import DataLoader

In [12]:
import torchvision
num_classes = 55
device = torch.device("cuda")

In [13]:
import kornia.augmentation as K
import torch.nn as nn

C:\Users\Nilansh Barotia\AppData\Local\Temp\ipykernel_10088\2275431781.py:86: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_1 = torch.cuda.amp.GradScaler()


Epoch 1/20 | Train Acc: 63.60% | Valid Acc: 87.67%
Epoch 2/20 | Train Acc: 88.50% | Valid Acc: 87.19%
Epoch 3/20 | Train Acc: 87.79% | Valid Acc: 91.33%
Epoch 4/20 | Train Acc: 86.03% | Valid Acc: 88.44%
Epoch 5/20 | Train Acc: 87.18% | Valid Acc: 88.05%
Epoch 6/20 | Train Acc: 88.66% | Valid Acc: 90.66%
Epoch 7/20 | Train Acc: 89.40% | Valid Acc: 92.10%
Epoch 8/20 | Train Acc: 90.27% | Valid Acc: 88.05%
Epoch 9/20 | Train Acc: 89.34% | Valid Acc: 94.51%
Epoch 10/20 | Train Acc: 90.40% | Valid Acc: 93.55%
Epoch 11/20 | Train Acc: 90.17% | Valid Acc: 90.37%
Epoch 12/20 | Train Acc: 93.13% | Valid Acc: 95.28%
Epoch 13/20 | Train Acc: 93.09% | Valid Acc: 95.47%
Epoch 14/20 | Train Acc: 94.19% | Valid Acc: 95.18%
Epoch 15/20 | Train Acc: 93.54% | Valid Acc: 95.86%
Epoch 16/20 | Train Acc: 95.73% | Valid Acc: 96.24%
Epoch 17/20 | Train Acc: 98.07% | Valid Acc: 97.78%
Epoch 18/20 | Train Acc: 98.72% | Valid Acc: 98.36%
Epoch 19/20 | Train Acc: 99.16% | Valid Acc: 98.55%
Epoch 20/20 | Train A

Now with batch size = 32

In [15]:
transform_2 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset_2 = dset(image_paths=image_paths, transform=transform_2)

ts = int(0.75*len(dataset_2))
vs = len(dataset_2) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_2, [ts, vs], generator)

train_loader_2 = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

valid_loader_2 = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False
)

model_2 = torchvision.models.resnet34(weights="DEFAULT")
model_2.fc = nn.Linear(
    model_2.fc.in_features,
    num_classes
)

model_2 = model_2.to(device)

for param in model_2.parameters():
    param.requires_grad = False

for param in model_2.fc.parameters():
    param.requires_grad = True

train_aug_2 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast=0.2,
        p=0.5
    ),
    K.RandomPlanckianJitter(
        mode="CIED",
        p=0.5
    )
).to(device)

targeted_aug_2 = K.AugmentationSequential(
    K.RandomRotation(
        degrees=10,
        p=0.5
    ),
    K.RandomAffine(
        degrees=0,
        scale=(0.9, 1.1),
        p=0.5
    ),
    K.RandomPerspective(
        distortion_scale=0.2,
        p=0.5
    ),
    K.ColorJiggle(
        brightness=0.2,
        contrast=0.2,
        p=0.5
    )
).to(device)

optimizer_2 = torch.optim.RMSprop(
    model_2.fc.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_2,
    max_lr=lr_head,
    epochs=20,
    steps_per_epoch=len(train_loader_2)
)

criterion = nn.CrossEntropyLoss()

scaler_2 = torch.cuda.amp.GradScaler()

for epoch in range(20):
    model_2.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_2:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_2(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_2(images[mask])

        optimizer_2.zero_grad()

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model_2(images)
            loss = criterion(outputs, labels)

        scaler_2.scale(loss).backward()

        scaler_2.step(optimizer_2)

        scaler_2.update()

        scheduler_2.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_2.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_2:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_2(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

C:\Users\Nilansh Barotia\AppData\Local\Temp\ipykernel_10088\772050278.py:86: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_2 = torch.cuda.amp.GradScaler()


Epoch 1/20 | Train Acc: 61.74% | Valid Acc: 88.25%
Epoch 2/20 | Train Acc: 91.20% | Valid Acc: 92.10%
Epoch 3/20 | Train Acc: 92.19% | Valid Acc: 90.27%
Epoch 4/20 | Train Acc: 89.98% | Valid Acc: 92.00%
Epoch 5/20 | Train Acc: 90.33% | Valid Acc: 93.35%
Epoch 6/20 | Train Acc: 92.32% | Valid Acc: 94.32%
Epoch 7/20 | Train Acc: 95.09% | Valid Acc: 91.91%
Epoch 8/20 | Train Acc: 93.93% | Valid Acc: 94.03%
Epoch 9/20 | Train Acc: 92.71% | Valid Acc: 95.28%
Epoch 10/20 | Train Acc: 94.57% | Valid Acc: 94.32%
Epoch 11/20 | Train Acc: 95.47% | Valid Acc: 93.35%
Epoch 12/20 | Train Acc: 94.99% | Valid Acc: 95.47%
Epoch 13/20 | Train Acc: 97.59% | Valid Acc: 95.57%
Epoch 14/20 | Train Acc: 96.43% | Valid Acc: 97.11%
Epoch 15/20 | Train Acc: 95.47% | Valid Acc: 97.21%
Epoch 16/20 | Train Acc: 97.94% | Valid Acc: 97.98%
Epoch 17/20 | Train Acc: 99.10% | Valid Acc: 98.27%
Epoch 18/20 | Train Acc: 99.29% | Valid Acc: 98.36%
Epoch 19/20 | Train Acc: 99.87% | Valid Acc: 98.55%
Epoch 20/20 | Train A